In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from scipy.stats import mannwhitneyu
from scipy.stats import levene

df_mec = {}
row_summary = {}
patient_ids = [d for d in os.listdir(os.path.join('..', 'Data', 'COVID-19-Wearables-MMD', 'S1')) if os.path.isdir(os.path.join('..', 'Data', 'COVID-19-Wearables-MMD', 'S1', d))]

df_statistic = pd.DataFrame(columns=['patient_id', 'mec', 'mannwhitney_statistic', 'mannwhitney_p_value', 'levene_statistic', 'levene_p_value'])
rows = []
for patient_id in [p for p in patient_ids if p not in ['AJ7TSV9','AS2MVDL']]:
    for mec in ['S1', 'S2', 'S3']:
        df_mec[mec] = [pd.read_csv(os.path.join(os.path.join('..', 'Data', 'COVID-19-Wearables-MMD', mec, patient_id, '1'), f)) 
                    for f in sorted(os.listdir(os.path.join('..', 'Data', 'COVID-19-Wearables-MMD', mec, patient_id, '1'))) if f.endswith('.csv')]
        
        df_sum = pd.read_csv(
            os.path.join('..', 'Data', 'COVID-19-Wearables-MMD',
                            f'summary_{mec}_{30}.csv')
        )

        row_summary[mec] = df_sum[
            (df_sum["file_id"] == patient_id) &
            (df_sum["dataset_iteration"] == 1)
        ]
        
    df_mec_particionado = {}

    for mec, lista_dfs in df_mec.items():
        rs = row_summary[mec]

        cols_idx = sorted(
            [c for c in rs.columns if c.startswith("index_")],
            key=lambda c: int(c.split("_")[1])
        )

        cortes = rs.iloc[0][cols_idx].dropna().astype(int).tolist()
        if not cortes:
            continue

        df_mec_particionado[mec] = []

        for df in lista_dfs:
            limites = [i for i in cortes]

            partes = [
                df.iloc[inicio:fim + 1].copy()
                for inicio, fim in zip(limites[:-1], limites[1:])
            ]

            df_mec_particionado[mec].append(partes)


        row_result = {
            'patient_id': patient_id,
            'mec': mec
        }
        segmentos = df_mec_particionado[mec][0]

        for i in range(len(segmentos) - 1):
            x = segmentos[i]['heartrate'].dropna()
            y = segmentos[i+1]['heartrate'].dropna()
            # x = segmentos[i]['target']
            # y = segmentos[i+1]['target']

            stat_mw, p_mw = mannwhitneyu(x, y, alternative='two-sided')
            stat_lv, p_lv = levene(x, y, center='median')

            row_result[f'mw_p_{i}_{i+1}'] = p_mw
            row_result[f'lev_p_{i}_{i+1}'] = p_lv
        rows.append(row_result)

df_statistic = pd.DataFrame(rows)


In [6]:
df_statistic[df_statistic['mec'] == 'S3']
# df_statistic.to_csv('statistic_results.csv', index=False)

,patient_id,mec,mw_p_0_1,lev_p_0_1,mw_p_1_2,lev_p_1_2,mw_p_2_3,lev_p_2_3,mw_p_3_4,lev_p_3_4
2,A0NVTRV,S3,0.000000e+00,8.517479e-115,0.000000e+00,9.102172e-168,0.000000e+00,1.820232e-09,4.494264e-157,3.800097e-281
5,AOYM4KG,S3,2.896423e-30,2.060490e-95,5.483821e-75,0.000000e+00,0.000000e+00,0.000000e+00,9.524926e-210,4.941172e-53
8,AUY8KYW,S3,1.395996e-213,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
11,AQC0L71,S3,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.701063e-112,0.000000e+00
14,AV2GF3B,S3,8.779746e-10,1.220134e-71,4.595683e-17,2.788885e-60,1.067181e-36,0.000000e+00,2.082767e-216,0.000000e+00
17,A1K5DRI,S3,3.870921e-71,0.000000e+00,0.000000e+00,6.601980e-204,3.518837e-88,4.076871e-287,0.000000e+00,2.089302e-281
20,AAXAA7Z,S3,5.644970e-129,0.000000e+00,1.031416e-97,0.000000e+00,0.000000e+00,0.000000e+00,4.232043e-252,0.000000e+00
23,AJWW3IY,S3,2.351492e-01,0.000000e+00,0.000000e+00,0.000000e+00,3.915987e-04,0.000000e+00,6.232037e-09,3.540749e-180
26,AJMQUVV,S3,0.000000e+00,1.226783e-09,0.000000e+00,0.000000e+00,4.701498e-03,0.000000e+00,9.431000e-289,0.000000e+00
29,ASFODQR,S3,0.000000e+00,2.015899e-53,0.000000e+00,1.451237e-34,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [110]:
alpha = 0.01

signals = {}
for row in df_statistic.itertuples(index=False):
    
    row_dict = row._asdict()
    mec = row_dict['mec']
    patient_id = row_dict['patient_id']

    if mec not in signals:
        signals[mec] = {}

    if patient_id not in signals[mec]:
        signals[mec][patient_id] = {
            'mannwhitney': [],
            'levene': []
        }

    for col, value in row_dict.items():
        
        if pd.isna(value):
            continue

        if col.startswith('mw_p_'):
            if value < alpha:
                signals[mec][patient_id]['mannwhitney'].append(True)
            else:
                signals[mec][patient_id]['mannwhitney'].append(False)
        
        if col.startswith('lev_p_'):
            if value < alpha:
                signals[mec][patient_id]['levene'].append(True)
            else:
                signals[mec][patient_id]['levene'].append(False)

In [114]:
count = {}
for mec, patients in signals.items():
    for patient_id, tests in patients.items():
        mw_signals = tests['mannwhitney']
        lev_signals = tests['levene']
        if False not in mw_signals or False not in lev_signals:
            count[mec] = count.get(mec, 0) + 1

print(f"Porcentagem de rejeição da hipótese nula:")
for mec, c in count.items():
    total = len(df_statistic[df_statistic['mec'] == mec])
    print(f"Total de pacientes para MEC {mec}: {total}, {c} rejeições")
    percentage = (c / total) * 100 if total > 0 else 0
    print(f"MEC {mec}: {percentage:.2f}%")


Porcentagem de rejeição da hipótese nula:
Total de pacientes para MEC S1: 30, 30 rejeições
MEC S1: 100.00%
Total de pacientes para MEC S2: 30, 30 rejeições
MEC S2: 100.00%
Total de pacientes para MEC S3: 30, 30 rejeições
MEC S3: 100.00%


In [107]:
count = {mec: {'mannwhitneyu': 0, 'levene': 0, 'both': 0} for mec in signals.keys()}
for mec, patients in signals.items():
    for patient_id, tests in patients.items():
        mw_signals = tests['mannwhitney']
        lev_signals = tests['levene']
        if False in mw_signals:
            count[mec]['mannwhitneyu'] = count[mec].get('mannwhitneyu', 0) + 1
        if False in lev_signals:
            count[mec]['levene'] = count[mec].get('levene', 0) + 1
        if False in mw_signals and False in lev_signals:
            count[mec]['both'] = count[mec].get('both', 0) + 1

print(f"Porcentagem de rejeição da hipótese nula:")
for mec, c in count.items():
    total = len(df_statistic[df_statistic['mec'] == mec])
    print(f"Total de pacientes para MEC {mec}: {total}, Mann-Whitney U: {c['mannwhitneyu']} rejeições, Levene: {c['levene']} rejeições, Ambos: {c['both']} rejeições")
    percentage_mw = (c['mannwhitneyu'] / total) * 100 if total > 0 else 0
    percentage_lev = (c['levene'] / total) * 100 if total > 0 else 0
    percentage_both = (c['both'] / total) * 100 if total > 0 else 0
    print(f"MEC {mec}: Mann-Whitney U: {percentage_mw:.2f}%, Levene: {percentage_lev:.2f}%, Ambos: {percentage_both:.2f}%")

Porcentagem de rejeição da hipótese nula:
Total de pacientes para MEC S1: 30, Mann-Whitney U: 3 rejeições, Levene: 5 rejeições, Ambos: 0 rejeições
MEC S1: Mann-Whitney U: 10.00%, Levene: 16.67%, Ambos: 0.00%
Total de pacientes para MEC S2: 30, Mann-Whitney U: 1 rejeições, Levene: 7 rejeições, Ambos: 0 rejeições
MEC S2: Mann-Whitney U: 3.33%, Levene: 23.33%, Ambos: 0.00%
Total de pacientes para MEC S3: 30, Mann-Whitney U: 3 rejeições, Levene: 9 rejeições, Ambos: 0 rejeições
MEC S3: Mann-Whitney U: 10.00%, Levene: 30.00%, Ambos: 0.00%


In [5]:
patient_id = 'AAXAA7Z'
df_mec = {}
row_summary = {}
for mec in ['S1', 'S2', 'S3']:
    df_mec[mec] = [pd.read_csv(os.path.join(os.path.join('..', 'Data', 'COVID-19-Wearables-MMD', mec, patient_id, '1'), f)) 
                   for f in sorted(os.listdir(os.path.join('..', 'Data', 'COVID-19-Wearables-MMD', mec, patient_id, '1'))) if f.endswith('.csv')]
    
    df_sum = pd.read_csv(
        os.path.join('..', 'Data', 'COVID-19-Wearables-MMD',
                        f'summary_{mec}_{30}.csv')
    )

    row_summary[mec] = df_sum[
        (df_sum["file_id"] == patient_id) &
        (df_sum["dataset_iteration"] == 1)
    ]

    

In [7]:
df_mec_particionado = {}

for mec, lista_dfs in df_mec.items():
    rs = row_summary[mec]

    cols_idx = sorted(
        [c for c in rs.columns if c.startswith("index_")],
        key=lambda c: int(c.split("_")[1])
    )

    cortes = rs.iloc[0][cols_idx].dropna().astype(int).tolist()
    if not cortes:
        continue

    if cortes[0] != 0:
        cortes = [0] + cortes

    df_mec_particionado[mec] = []

    for df in lista_dfs:
        limites = [i for i in cortes if i < len(df)]
        if limites[-1] != len(df) - 1:
            limites.append(len(df) - 1)

        partes = [
            df.iloc[inicio:fim + 1].copy()
            for inicio, fim in zip(limites[:-1], limites[1:])
        ]
        df_mec_particionado[mec].append(partes)